In [ ]:
# Cell 1: Install arc-agi from competition wheels (offline, works in Phase A and B)
import subprocess, sys
wheels = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-index', f'--find-links={wheels}',
    'arc-agi', 'python-dotenv'
])
print('[Cell1] arc-agi installed from competition wheels')

In [ ]:
# Cell 2: Write VericodingAgent to /tmp/my_agent.py
agent_code = """import os, sys, random, itertools
import numpy as np
import torch
import torch.nn as nn
from agents.agent import Agent
from arcengine import GameAction


# ─── TTT Network ────────────────────────────────────────────────────────
class TTTNet(nn.Module):
    """Tiny world model: grid -> action scores (7-D). Trained online."""
    def __init__(self, n_actions=7):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(4)
        self.fc = nn.Linear(32*4*4, n_actions)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.to(self.device)
        self.opt = torch.optim.Adam(self.parameters(), lr=0.005)

    def forward(self, grid_t):
        """grid_t: (B, H, W) int -> scores (B, n_actions) [0,1]"""
        x = grid_t.unsqueeze(1).float() / 10.0
        x = x.to(self.device)
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = self.pool(x)
        x = x.flatten(1)
        return torch.sigmoid(self.fc(x))

    def train_step(self, grids, actions, targets):
        """grids: (B, H, W), actions: (B,) int, targets: (B,) float"""
        self.train()
        self.zero_grad()
        scores = self.forward(grids)  # (B, n_actions)
        batch_idx = torch.arange(len(actions), device=self.device)
        pred = scores[batch_idx, actions]
        loss = nn.functional.binary_cross_entropy(pred, targets.to(self.device))
        loss.backward()
        self.opt.step()
        return loss.item()

    @torch.no_grad()
    def predict(self, grid, valid_actions):
        """grid: (H, W) -> dict[action, score] for valid actions"""
        self.eval()
        if not valid_actions:
            return {}
        grid_t = torch.asarray(grid, dtype=torch.int32).unsqueeze(0)  # (1, H, W)
        scores = self.forward(grid_t)[0]  # (n_actions,)
        return {a: float(scores[a]) for a in valid_actions if a < len(scores)}


# ─── Main Agent ─────────────────────────────────────────────────────────
class VericodingAgent(Agent):
    """DenseExplorer + TTT: learns action values online per game."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._step = 0
        self._phase = 'graph'       # graph -> dense -> fallback
        self._phase_step = 0
        self._visited = set()
        self._last_grid = None
        self._last_action = None
        self._stall = 0
        self._prev_act = None
        self._sx, self._sy = 0, 0

        # D4 Zobrist table
        rng = np.random.RandomState(42)
        self._zob = rng.randint(0, 2**63-1, size=(64, 64, 16), dtype=np.uint64)

        # TTT
        self._ttt = TTTNet()
        self._buffer = []  # [(grid, action, reward)]
        self._ttt_train_every = 5
        self._ttt_used = 0
        self._ttt_good = 0
        print(f'[init] TTT device={self._ttt.device}')

    def _hash(self, grid):
        h, w = grid.shape[:2]
        best = np.uint64(2**64-1)
        for rot in range(4):
            for flip in (False, True):
                g = np.rot90(grid, rot)
                if flip: g = np.fliplr(g)
                ph = np.bitwise_xor.reduce(
                    self._zob[:g.shape[0], :g.shape[1]] * (g[..., None] > 0), axis=2)
                hv = int(np.bitwise_xor.reduce(ph))
                if hv < best: best = hv
        return int(best)

    def choose_action(self, frames, latest_frame) -> GameAction:
        try: return self._strategy(latest_frame)
        except Exception as e:
            print(f'[err] {e}')
            return self._rand_act(latest_frame)

    def _strategy(self, latest_frame) -> GameAction:
        self._step += 1
        self._phase_step += 1
        grid = np.array(latest_frame.frame[0], dtype=np.int32)
        avail = list(latest_frame.available_actions) if latest_frame.available_actions else []
        if not avail: return GameAction(1)
        h, w = grid.shape[:2]

        # Hash state
        hsh = self._hash(grid)
        is_new = hsh not in self._visited
        self._visited.add(hsh)

        # TTT: record experience from LAST step
        if self._last_grid is not None and self._last_action is not None:
            changed = not np.array_equal(grid, self._last_grid) or is_new
            reward = 1.0 if changed else 0.0
            self._buffer.append((self._last_grid.copy(), self._last_action, reward))
            if len(self._buffer) > 300:
                self._buffer.pop(0)

            # Train TTT every N steps
            if len(self._buffer) >= 5 and self._step % self._ttt_train_every == 0:
                batch = random.sample(self._buffer, min(32, len(self._buffer)))
                grids_np = np.array([b[0] for b in batch])
                acts = torch.tensor([b[1] for b in batch], dtype=torch.long)
                tgts = torch.tensor([b[2] for b in batch], dtype=torch.float32)
                gpu = torch.device(self._ttt.device)
                loss = self._ttt.train_step(
                    torch.asarray(grids_np, device=gpu),
                    acts.to(gpu),
                    tgts.to(gpu),
                )

        self._last_grid = grid.copy()

        # Stall detection
        if self._last_action is not None:
            # Check if stall (same grid after action)
            pass
        self._stall = 0

        # Phase transitions
        if self._phase_step > 50 or (self._stall > 5):
            if self._phase == 'graph':    self._phase = 'dense';  self._phase_step = 0; self._sx, self._sy = 0, 0
            elif self._phase == 'dense':  self._phase = 'fallback'; self._phase_step = 0

        # Separate action types
        complex_a = [a for a in avail if hasattr(a, 'is_complex') and a.is_complex()]
        simple_a  = [a for a in avail if a not in complex_a]
        has_cx = len(complex_a) > 0
        nz = np.count_nonzero(grid)

        # TTT scores for action selection
        ttt_scores = self._ttt.predict(grid, range(7)) if len(self._buffer) >= 5 else {}

        # --- GRAPH PHASE: click non-zero cells systematically ---
        if self._phase == 'graph' and has_cx:
            flat = grid.flatten()
            nz_idx = np.where(flat > 0)[0]
            if len(nz_idx) > 0:
                idx = nz_idx[self._phase_step % len(nz_idx)]
                y, x = divmod(int(idx), w)
                a = GameAction(6)
                a.set_data({'x': x, 'y': y})
                self._last_action = 6
                return a

        # --- DENSE PHASE: scan grid, use TTT to guide clicks ---
        if self._phase == 'dense' and has_cx:
            x, y = self._sx, self._sy
            self._sx += 1
            if self._sx >= w: self._sx = 0; self._sy += 1
            if self._sy >= h: self._sy = 0
            if 0 <= y < h and 0 <= x < w and grid[y, x] > 0:
                cx_click = 6
                # TTT says click is good?
                ttt_score = ttt_scores.get(cx_click, 0.5)
                if ttt_score > 0.3 or self._step < 10:
                    a = GameAction(6)
                    a.set_data({'x': x, 'y': y})
                    self._last_action = 6
                    if ttt_score > 0.3: self._ttt_used += 1
                    return a
            return self._rand_act(latest_frame)

        # --- TTT: if we have trained model, use it ---
        if len(self._buffer) >= 10 and ttt_scores:
            # Pick action with highest TTT score
            best_a = max(avail, key=lambda a: ttt_scores.get(a if isinstance(a, int) else 6, 0.5))
            if isinstance(best_a, int):
                act_type = best_a
            else:
                act_type = 6
            ttt_val = ttt_scores.get(act_type, 0.5)
            if ttt_val > 0.5:
                self._ttt_used += 1
                a = GameAction(act_type)
                if a.is_complex():
                    y = int(random.randint(0, h-1)) if h > 0 else 0
                    x = int(random.randint(0, w-1)) if w > 0 else 0
                    a.set_data({'x': x, 'y': y})
                self._last_action = act_type
                return a

        # --- FALLBACK: cycle actions ---
        if self._prev_act is not None and len(simple_a) > 1:
            others = [a for a in simple_a if a != self._prev_act]
            if others: chosen = random.choice(others); self._prev_act = chosen
            else: chosen = random.choice(simple_a); self._prev_act = chosen
        else:
            chosen = random.choice(simple_a) if simple_a else 1
            self._prev_act = chosen
        a = GameAction(chosen)
        if a.is_complex():
            a.set_data({'x': w//2, 'y': h//2})
        self._last_action = chosen if isinstance(chosen, int) else 6
        return a

    def _rand_act(self, latest_frame) -> GameAction:
        avail = list(getattr(latest_frame, 'available_actions', None) or [])
        if not avail: return GameAction(1)
        at = random.choice(avail)
        a = GameAction(at)
        if a.is_complex():
            a.set_data({'x': random.randint(0,63), 'y': random.randint(0,63)})
        return a
"""

with open("/tmp/my_agent.py", "w") as f:
    f.write(agent_code)
print('[Cell2] VericodingAgent with TTT written')


In [ ]:
# Cell 3: Phase B — Competition Rerun (gateway + framework)
import os, subprocess, shutil, sys

if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell3] Phase B: competition rerun detected')

    # Wait for gateway sidecar
    subprocess.run([
        'curl', '--fail', '--retry', '999', '--retry-all-errors',
        '--retry-delay', '5', '--retry-max-time', '600',
        'http://gateway:8001/api/games'
    ], check=True)
    print('[Cell3] gateway ready')

    # Copy framework from competition data
    comp = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3'
    fw_src = f'{comp}/ARC-AGI-3-Agents'
    fw_dst = '/kaggle/working/ARC-AGI-3-Agents'
    if os.path.exists(fw_src):
        shutil.copytree(fw_src, fw_dst, dirs_exist_ok=True)
    elif os.path.exists(f'{comp}/arc_agi_3_agents'):
        shutil.copytree(f'{comp}/arc_agi_3_agents', fw_dst, dirs_exist_ok=True)
    print(f'[Cell3] framework copied to {fw_dst}')

    # Install agent into framework
    agents_dir = f'{fw_dst}/agents'
    shutil.copy('/tmp/my_agent.py', f'{agents_dir}/my_agent.py')

    # Register VericodingAgent in framework __init__
    init_content = """from agents.my_agent import VericodingAgent

AVAILABLE_AGENTS = {
    "vericoding": VericodingAgent,
}
"""
    with open(f'{agents_dir}/__init__.py', 'w') as f:
        f.write(init_content)

    # Run framework
    sys.path.insert(0, fw_dst)
    result = subprocess.run(
        [sys.executable, 'main.py', '--agent', 'vericoding'],
        cwd=fw_dst, capture_output=False
    )
    print(f'[Cell3] framework exit code: {result.returncode}')
else:
    print('[Cell3] Phase A: skipping (no competition rerun)')

In [ ]:
# Cell 4: Phase A — Write dummy submission.parquet (required to enable Submit button)
import os
import pandas as pd

if not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell4] Phase A: writing dummy submission.parquet')
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    ).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('[Cell4] submission.parquet written — click Submit to Competition on Kaggle')
else:
    print('[Cell4] Phase B: submission.parquet handled by framework gateway')